In [ ]:
import concurrent.futures
import os
import re
import subprocess

import pandas as pd
from dotenv import load_dotenv
from loguru import logger

load_dotenv()

In [ ]:
PASSWORD_STORE_DIR = os.path.expanduser("~/.password-store")
EMAIL_PATTERN = r'([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})'
URL_PATTERN = r'/([^/]+\.[^/]+)/'

In [ ]:
def list_gpg_files(folder_path):
    gpg_files = []

    # Walk through the directory tree
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".gpg"):
                # Append the full path of the .gpg file to the list
                gpg_files.append(os.path.join(root, file))

    return gpg_files

In [ ]:
def extract_password_details(entry_name):
    # Retrieve the GPG passphrase from environment variables
    passphrase = os.getenv("GPG_PASSPHRASE")
    if not passphrase:
        raise ValueError("GPG_PASSPHRASE environment variable is not set")

    # Extract possible URL from the entry name
    url = extract_url(entry_name)

    # Decrypt the password file using GPG
    password_details = decrypt_password_file(entry_name, passphrase)
    if not password_details:
        return None

    # Process decrypted lines to extract relevant details
    password = password_details[0]
    email = extract_email(password_details)
    username = extract_username(password_details)
    notes = extract_notes(password_details)

    # Construct the details dictionary
    details = {
        "name": url,
        "url": url,
        "email": email,
        "username": username,
        "password": password,
        "note": "\n".join(notes),
        "totp": "",  # If TOTP is available, extract it
        "vault": "Personal",
    }
    return details

def extract_url(entry_name):
    possible_url = re.search(URL_PATTERN, entry_name)
    return possible_url.group(1) if possible_url else ""

def decrypt_password_file(entry_name, passphrase):
    result = subprocess.run(
        [
            "gpg",
            "--batch",
            "--yes",
            "--pinentry-mode",
            "loopback",
            "--passphrase",
            passphrase,
            "--decrypt",
            entry_name,
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        logger.error(f"Failed to decrypt password for {entry_name}")
        logger.error(result.stderr)
        return None
    return result.stdout.splitlines()

def extract_email(password_details):
    login_line = next((line for line in password_details[1:] if "login:" in line), "")
    email_match = re.search(EMAIL_PATTERN, login_line)
    return email_match.group(1) if email_match else ""

def extract_username(password_details):
    username_line = next((line for line in password_details[1:] if "username:" in line), "")
    return username_line.split(":")[1].strip() if username_line else ""

def extract_notes(password_details):
    return [line for line in password_details[1:] if not any(keyword in line for keyword in ["login:", "username:"])]

In [ ]:
extract_password_details("/home/fbossiere/.password-store/panzoid.com/francois.bossiere@polynom.io.gpg")

In [ ]:
# Define the template DataFrame
df_template = pd.DataFrame(
    columns=["name", "url", "email", "username", "password", "note", "totp", "vault"]
)
password_entries = list_gpg_files(PASSWORD_STORE_DIR)
logger.info(f"Found {len(password_entries)} password entries.")

if not password_entries:
    raise Exception(
        "Failed to retrieve passwords. Ensure you have GnuPG set up correctly."
    )

In [ ]:
# Parallel processing using ThreadPoolExecutor
with concurrent.futures.ProcessPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(extract_password_details, password_entries))

# Filter out None results (in case extract_password_details returns None)
filtered_results = [result for result in results if result]

# Convert the data into a DataFrame
df = pd.DataFrame(filtered_results)

# Ensure it follows the template format
df = pd.concat([df_template, df], ignore_index=True)
df.head()

In [ ]:
# Write the DataFrame to a CSV
df.to_csv("../data/passwords_export.csv", index=False, quoting=1)